Acknowledgements and references:

- [My previous public notebook: 5800.55 LB NeuroGolf Task-Level ONNX Blend](https://www.kaggle.com/code/kojimar/5800-55-lb-neurogolf-task-level-onnx-blend) - structure/reference for this readable task-level rebuild notebook.
- [6042.85 Per-Task Hand-Built ONNX Solvers](https://www.kaggle.com/code/octaviograu/6042-85-per-task-hand-built-onnx-solvers) - main public base bundle reproduced as the starting point.
- [6029.09 LB NeuroGolf all-task ONNX solution](https://www.kaggle.com/code/jsrdcht/6029-09-lb-neurogolf-all-task-onnx-solution) - important all-task public reference during task-level attribution.

Thanks to the NeuroGolf public notebook and discussion contributors. The companion dataset only contains the two zip files needed to reproduce `/kaggle/working/submission.zip`.


In [ ]:
from __future__ import annotations

import re
import zipfile
from pathlib import Path


TASK_COUNT = 400
TASK_RE = re.compile(r"^task(\d{3})\.onnx$")

WORK_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("working")
WORK_ROOT.mkdir(parents=True, exist_ok=True)
SUBMISSION_ZIP = WORK_ROOT / "submission.zip"


In [ ]:
def valid_task_name(name: str) -> bool:
    match = TASK_RE.match(Path(name).name)
    return bool(match and 1 <= int(match.group(1)) <= TASK_COUNT)


def task_name(task_num: int) -> str:
    return f"task{task_num:03d}.onnx"


def read_task_zip(zip_path: Path) -> dict[str, bytes]:
    tasks: dict[str, bytes] = {}
    with zipfile.ZipFile(zip_path) as zf:
        for entry in zf.infolist():
            if entry.is_dir():
                continue
            name = Path(entry.filename).name
            if valid_task_name(name):
                tasks[name] = zf.read(entry)
    return tasks


def read_task_dir(directory: Path) -> dict[str, bytes]:
    tasks: dict[str, bytes] = {}
    for path in directory.rglob("*.onnx"):
        name = path.name
        if valid_task_name(name):
            tasks[name] = path.read_bytes()
    return tasks


def read_tasks(path: Path) -> dict[str, bytes]:
    if path.is_file():
        return read_task_zip(path)
    return read_task_dir(path)


In [ ]:
def candidate_roots() -> list[Path]:
    roots = [Path.cwd()]
    if Path("/kaggle/input").exists():
        roots.append(Path("/kaggle/input"))
    candidates: list[Path] = []
    for root in roots:
        if not root.exists():
            continue
        candidates.append(root)
        candidates.extend(path for path in root.rglob("*") if path.is_dir())
    return candidates


def find_asset_paths() -> tuple[Path, Path]:
    for root in candidate_roots():
        base_zip = root / "base_submission.zip"
        overrides_zip = root / "overrides.zip"
        if base_zip.exists() and overrides_zip.exists():
            return base_zip, overrides_zip

        base_dir = root / "base_submission"
        overrides_dir = root / "overrides"
        if base_dir.is_dir() and overrides_dir.is_dir():
            return base_dir, overrides_dir

    raise FileNotFoundError(
        "Attach the companion dataset containing base_submission/overrides or base_submission.zip/overrides.zip."
    )


BASE_ASSET, OVERRIDES_ASSET = find_asset_paths()


In [ ]:
base_tasks = read_tasks(BASE_ASSET)
override_tasks = read_tasks(OVERRIDES_ASSET)
base_tasks.update(override_tasks)


In [ ]:
with zipfile.ZipFile(SUBMISSION_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    for task_num in range(1, TASK_COUNT + 1):
        name = task_name(task_num)
        zf.writestr(name, base_tasks[name])
